# DisasterScout — Model Training
This notebook covers the ML foundation for DisasterScout (Phase 0). It sets up the PyTorch U-Net model, loads the xBD dataset, and provides the training loop skeleton.

In [ ]:
!pip install segmentation-models-pytorch torchgeo rasterio albumentations geopandas

In [ ]:
import os
import pathlib
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import segmentation_models_pytorch as smp

In [ ]:
# check for GPU
cuda_available = torch.cuda.is_available()
print(f"CUDA available: {cuda_available}")
if cuda_available:
    print(torch.cuda.get_device_name(0))
device = torch.device('cuda' if cuda_available else 'cpu')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# User should change this to their specific Drive path
XBD_PATH = '/content/drive/MyDrive/xBD'

In [ ]:
class xBDDataset(torch.utils.data.Dataset):
    """
    xBD Dataset class (custom implementation without TorchGeo).
    Note: Using concatenated 6-channel input, not Siamese architecture, for speed.
    """
    def __init__(self, root, split='train', disaster_types=None):
        self.root = root
        self.split = split
        # xBD data generally looks like:
        # xBD/<disaster_type>/images/<name>_pre_disaster.png
        # xBD/<disaster_type>/images/<name>_post_disaster.png
        # xBD/<disaster_type>/labels/<name>_post_disaster.json (damage labels)

    def __len__(self):
        return 0

    def __getitem__(self, idx):
        # Return dict format:
        # {"image": tensor(6, 512, 512), "mask": tensor(512, 512), "name": str}
        pass

In [ ]:
def show_sample(pre_img, post_img, mask):
    # Visualize a sample pre/post and its mask side-by-side
    # using damage colors: no_change=gray, flood=blue, structural=red, road=orange
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    axes[0].imshow(pre_img)
    axes[0].set_title('Pre Disaster')
    axes[1].imshow(post_img)
    axes[1].set_title('Post Disaster')
    axes[2].imshow(mask)
    axes[2].set_title('Damage Mask')
    plt.show()

In [ ]:
# Build Model
model = smp.Unet(
    encoder_name='resnet50',
    encoder_weights='imagenet',
    in_channels=6,
    classes=4,
    activation=None
)
model = model.to(device)
print("Model summary:\n", model)

In [ ]:
# Training loop skeleton
NUM_EPOCHS = 25

# DataLoader setup (batch_size=4, num_workers=2)
# train_loader = DataLoader(..., batch_size=4, num_workers=2)

# Loss with class weights to penalize no_change class
class_weights = torch.tensor([0.1, 1.0, 1.5, 1.5]).to(device)
criterion = nn.CrossEntropyLoss(weight=class_weights)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

for epoch in range(NUM_EPOCHS):
    model.train()
    # for batch in train_loader:
        # TODO: forward pass
        # TODO: compute loss
        # TODO: backward + optimizer step
        # TODO: log train loss
        # pass
    
    print(f"Epoch {epoch+1} / {NUM_EPOCHS} complete.")

In [ ]:
# Save checkpoint
torch.save(model.state_dict(), '/content/drive/MyDrive/disasterscout_checkpoint.pth')
print("Saved to Drive")